In [5]:
!rm -rf /kaggle/working/LIBERO_FORK_RUNNER /kaggle/working/LIBERO
!git clone https://github.com/evanmanolis/LIBERO_FORK.git /kaggle/working/LIBERO_FORK_RUNNER
!grep -n "config_dict" /kaggle/working/LIBERO_FORK_RUNNER/kaggle_libero_runner.py


Cloning into '/kaggle/working/LIBERO_FORK_RUNNER'...
remote: Enumerating objects: 1445, done.
remote: Counting objects: 100% (602/602), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 1445 (delta 379), reused 385 (delta 369), pack-reused 843 (from 1)
Receiving objects: 100% (1445/1445), 310.15 MiB | 39.23 MiB/s, done.
Resolving deltas: 100% (657/657), done.
Updating files: 100% (1123/1123), done.
439:    config_dict = asdict(cfg)
440:    for key, value in list(config_dict.items()):
442:            config_dict[key] = sorted(value)
451:        "config": config_dict,


In [6]:
!rm -rf /kaggle/working/LIBERO_FORK_RUNNER /kaggle/working/LIBERO
!git clone https://github.com/evanmanolis/LIBERO_FORK.git /kaggle/working/LIBERO_FORK_RUNNER
!grep -n "normalize_hydra_overrides" /kaggle/working/LIBERO_FORK_RUNNER/kaggle_libero_runner.py


Cloning into '/kaggle/working/LIBERO_FORK_RUNNER'...
remote: Enumerating objects: 1445, done.
remote: Counting objects: 100% (602/602), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 1445 (delta 379), reused 385 (delta 369), pack-reused 843 (from 1)
Receiving objects: 100% (1445/1445), 310.15 MiB | 31.16 MiB/s, done.
Resolving deltas: 100% (657/657), done.
Updating files: 100% (1123/1123), done.
116:def normalize_hydra_overrides(overrides: Sequence[str]) -> List[str]:
871:        extra_overrides=normalize_hydra_overrides(args.extra_override),


In [ ]:
# LIBERO_FORK + Rational Activation Runner (single Kaggle cell)
import json
import pathlib
import shutil
import subprocess
import sys
import shlex

# ---------------------------
# User controls
# ---------------------------
RUN_SETUP = True
RUN_TASKS = True

# Your fork
REPO_URL = "https://github.com/evanmanolis/LIBERO_FORK.git"

# Runner bootstrap clone (just to get your updated runner script)
RUNNER_REPO_DIR = pathlib.Path("/kaggle/working/LIBERO_FORK_RUNNER")

# The actual training repo path used by kaggle_libero_runner.py
TARGET_LIBERO_DIR = pathlib.Path("/kaggle/working/LIBERO")

RUN_STAGE = "full"  # canary0 | canary1 | canary2 | full
TARGET_TASK_IDS = [0]  # zero-based task IDs
EVAL_NUM_PROCS = 1
REBUILD_ENV = False
SNAPSHOT_PERIOD_SEC = 1500
PACKAGE_NAME = "libero_object_tasks_3_4_rational"

# Safe rational activation (low degree, clipped input)
EXTRA_OVERRIDES = [
    "eval.eval_every=10",
    "policy.transformer_ffn_activation=rational",
    "policy.transformer_ffn_activation_kwargs.numerator_degree=3",
    "policy.transformer_ffn_activation_kwargs.denominator_degree=2",
    "policy.transformer_ffn_activation_kwargs.input_clip=10.0",
    "policy.transformer_ffn_activation_kwargs.init=identity",
    
    "policy.language_encoder.network_kwargs.activation=rational",
    "policy.language_encoder.network_kwargs.activation_kwargs.numerator_degree=3",
    "policy.language_encoder.network_kwargs.activation_kwargs.denominator_degree=2",
    "policy.language_encoder.network_kwargs.activation_kwargs.input_clip=10.0",
    "policy.language_encoder.network_kwargs.activation_kwargs.init=identity",

    "policy.policy_head.network_kwargs.hidden_activation=rational",
    "policy.policy_head.network_kwargs.hidden_activation_kwargs.numerator_degree=3",
    "policy.policy_head.network_kwargs.hidden_activation_kwargs.denominator_degree=2",
    "policy.policy_head.network_kwargs.hidden_activation_kwargs.input_clip=10.0",
    "policy.policy_head.network_kwargs.hidden_activation_kwargs.init=identity",

    "policy.extra_activation=rational",
    "policy.extra_activation_kwargs.numerator_degree=3",
    "policy.extra_activation_kwargs.denominator_degree=2",
    "policy.extra_activation_kwargs.input_clip=10.0",
    "policy.extra_activation_kwargs.init=identity",
]

# ---------------------------
# Helpers
# ---------------------------
def norm_git_url(url: str) -> str:
    return url.strip().rstrip("/").removesuffix(".git").lower()

def run(cmd, check=True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run(cmd, check=check)

# ---------------------------
# 1) Get updated runner from your fork
# ---------------------------
if RUNNER_REPO_DIR.exists():
    run(["git", "-C", str(RUNNER_REPO_DIR), "pull", "--ff-only"], check=False)
else:
    run(["git", "clone", REPO_URL, str(RUNNER_REPO_DIR)], check=True)

runner_script = RUNNER_REPO_DIR / "kaggle_libero_runner.py"
if not runner_script.exists():
    raise RuntimeError(f"Runner script not found: {runner_script}")

# ---------------------------
# 2) Ensure /kaggle/working/LIBERO is your fork (not upstream)
# ---------------------------
if RUN_SETUP and TARGET_LIBERO_DIR.exists():
    origin = ""
    try:
        origin = subprocess.check_output(
            ["git", "-C", str(TARGET_LIBERO_DIR), "remote", "get-url", "origin"],
            text=True
        ).strip()
    except Exception:
        origin = ""
    if norm_git_url(origin) != norm_git_url(REPO_URL):
        print(f"Removing stale repo at {TARGET_LIBERO_DIR} (origin was: {origin})")
        shutil.rmtree(TARGET_LIBERO_DIR)

# ---------------------------
# 3) Run setup/tasks with rational overrides
# ---------------------------
if not (RUN_SETUP or RUN_TASKS):
    raise SystemExit("Set RUN_SETUP and/or RUN_TASKS to True.")

cmd = [sys.executable, str(runner_script)]
if RUN_SETUP:
    cmd.append("--setup")
if RUN_TASKS:
    cmd.append("--run")

cmd += [
    "--repo-url", REPO_URL,
    "--run-stage", RUN_STAGE,
    "--task-ids", ",".join(str(x) for x in TARGET_TASK_IDS),
    "--eval-num-procs", str(EVAL_NUM_PROCS),
    "--snapshot-period-sec", str(SNAPSHOT_PERIOD_SEC),
    "--package-name", PACKAGE_NAME,
]

if REBUILD_ENV:
    cmd.append("--rebuild-env")

for ov in EXTRA_OVERRIDES:
    cmd += ["--extra-override", ov]

run(cmd, check=True)

# ---------------------------
# 4) Verify rational is in saved run configs
# ---------------------------
if RUN_TASKS:
    bundle_dir = pathlib.Path(f"/kaggle/working/{PACKAGE_NAME}")
    summary_path = bundle_dir / "summary.json"
    zip_path = pathlib.Path(f"/kaggle/working/{PACKAGE_NAME}.zip")

    if not summary_path.exists():
        raise RuntimeError(f"Missing summary: {summary_path}")

    summary = json.loads(summary_path.read_text())
    print("Errors from summary:", summary.get("errors", []))

    task_dirs = sorted(bundle_dir.glob("task*"))
    if not task_dirs:
        raise RuntimeError(f"No task directories found in {bundle_dir}")

    for task_dir in task_dirs:
        cfg_path = task_dir / "config.json"
        if not cfg_path.exists():
            print(f"[warn] missing {cfg_path} (task may have failed early)")
            continue

        cfg = json.loads(cfg_path.read_text())
        p = cfg.get("policy", {})

        tf_act = p.get("transformer_ffn_activation")
        lang_act = p.get("language_encoder", {}).get("network_kwargs", {}).get("activation")
        head_act = p.get("policy_head", {}).get("network_kwargs", {}).get("hidden_activation")

        print(f"{task_dir.name}: transformer_ffn={tf_act}, language={lang_act}, policy_head={head_act}")

        assert tf_act == "rational", f"{task_dir.name}: transformer_ffn_activation != rational"
        assert lang_act == "rational", f"{task_dir.name}: language activation != rational"
        assert head_act == "rational", f"{task_dir.name}: policy head hidden activation != rational"

    print("Rational activation verified in saved configs.")
    print("Bundle:", bundle_dir)
    print("Zip   :", zip_path, "| exists:", zip_path.exists())


$ git -C /kaggle/working/LIBERO_FORK_RUNNER pull --ff-only
Already up to date.
$ /usr/bin/python3 /kaggle/working/LIBERO_FORK_RUNNER/kaggle_libero_runner.py --setup --run --repo-url https://github.com/evanmanolis/LIBERO_FORK.git --run-stage full --task-ids 0 --eval-num-procs 1 --snapshot-period-sec 1500 --package-name libero_object_tasks_3_4_rational --extra-override eval.eval_every=10 --extra-override policy.transformer_ffn_activation=rational --extra-override policy.transformer_ffn_activation_kwargs.numerator_degree=3 --extra-override policy.transformer_ffn_activation_kwargs.denominator_degree=2 --extra-override policy.transformer_ffn_activation_kwargs.input_clip=10.0 --extra-override policy.transformer_ffn_activation_kwargs.init=identity --extra-override policy.language_encoder.network_kwargs.activation=rational --extra-override policy.language_encoder.network_kwargs.activation_kwargs.numerator_degree=3 --extra-override policy.language_encoder.network_kwargs.activation_kwargs.denomi